# Phase 2 · Task 5 — Feature Engineering & Feature Selection

**Project:** Quantum Optimization for Formula 1 Race Strategy

This notebook turns the cleaned per-lap dataset produced by **Task 4 (Data Engineering & EDA)**
into the model-ready feature matrix consumed by **Task 6 (Classical Machine Learning)** and,
downstream, by Tasks 7–8 (Deep Learning / Quantum Machine Learning).

### Input
| | |
|---|---|
| Source | `../phase1_task4_data_engineering/outputs/clean/fastf1_laps_clean.csv` |
| Fallback | runs `../phase1_task4_data_engineering/run_eda.py` if the file is absent |

### Output (the only two files this notebook writes)
| File | Purpose |
|---|---|
| `outputs/f1_features_selected.csv` | Final feature-selected modelling matrix + targets |
| `outputs/feature_metadata.json` | Selected feature lists, provenance, and the scaling / splitting contract for Tasks 6–8 |

No figures, diagrams, entity tables or intermediate datasets are produced — everything
below stays in this notebook.

### Dependencies
`pandas`, `numpy`, `scikit-learn` (all already in Task 4's `requirements.txt`).
Variance-inflation factors are computed directly from `sklearn.LinearRegression`,
so **no `statsmodels` dependency is introduced**.

### Design commitments
1. **Reuse, don't duplicate.** Column semantics come from Task 4's `f1data.schemas`;
   the cleaned data is read in place rather than re-copied.
2. **Causality.** Every history-derived feature uses only laps *strictly before* the lap
   being predicted (`shift(1)`), so nothing leaks from the future.
3. **No exact collinearity by construction.** Features are built on a basis that avoids
   exact linear identities, so VIF prunes genuine redundancy rather than arbitrarily
   deleting physically meaningful variables.
4. **Scaling is deferred to the modelling task.** Task 5 exports *unscaled* features and
   records which columns need scaling, so Tasks 6–8 can fit the scaler **inside** their
   cross-validation folds. Scaling here would leak test-fold statistics into training.

## 1 · Setup and path resolution

Locate the Task 4 module and its cleaned output by walking up from the notebook's
location, so the notebook works from a clone regardless of the absolute path.

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.linear_model import LassoCV, LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160, "display.max_columns", 60)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def find_project_root(start: Path) -> Path:
    '''Walk upwards until the folder containing the Phase-1 task packages is found.'''
    for candidate in [start, *start.parents]:
        if (candidate / "phase1_task4_data_engineering").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate 'phase1_task4_data_engineering'. Run this notebook from "
        "inside the project tree."
    )


HERE = Path.cwd()
ROOT = find_project_root(HERE)
TASK4 = ROOT / "phase1_task4_data_engineering"
CLEAN_CSV = TASK4 / "outputs" / "clean" / "fastf1_laps_clean.csv"
OUT_DIR = (ROOT / "phase2_task5_feature_engineering" / "outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Reuse Task 4's canonical column semantics instead of redefining them here.
sys.path.insert(0, str(TASK4 / "src"))
from f1data.schemas import ALL_SCHEMAS, ColKind  # noqa: E402

LAPS_SCHEMA = ALL_SCHEMAS["fastf1_laps"]

print(f"project root : {ROOT}")
print(f"task 4       : {TASK4.name}")
print(f"output dir   : {OUT_DIR.relative_to(ROOT)}")

project root : /home/claude/work/phase 1-4
task 4       : phase1_task4_data_engineering
output dir   : phase2_task5_feature_engineering/outputs


## 2 · Load the cleaned dataset from Task 4

We consume Task 4's output directly. If `outputs/clean/` has not been generated in this
clone, we invoke Task 4's own driver rather than re-implementing its cleaning logic —
this keeps a single source of truth for how the data is cleaned.

In [2]:
if not CLEAN_CSV.exists():
    print("Cleaned data not found - regenerating via Task 4's pipeline ...")
    result = subprocess.run(
        [sys.executable, "run_eda.py"], cwd=TASK4,
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Task 4 pipeline failed:\n{result.stderr[-2000:]}")
    print("Task 4 pipeline completed.")

laps = pd.read_csv(CLEAN_CSV)
print(f"Loaded {CLEAN_CSV.relative_to(ROOT)}")
print(f"shape: {laps.shape[0]} laps x {laps.shape[1]} columns")
laps.head()

Loaded phase1_task4_data_engineering/outputs/clean/fastf1_laps_clean.csv
shape: 550 laps x 20 columns


,Driver,Team,LapNumber,LapTime,Stint,Compound,TyreLife,FreshTyre,Sector1Time,Sector2Time,Sector3Time,SpeedFL,SpeedST,TrackStatus,AirTemp,TrackTemp,Humidity,Rainfall,WindSpeed,IsPersonalBest
0,HAM,MERCEDES,1,90.200,1,SOFT,1,True,27.033,37.824,25.343,297.8,320.2,1,24.3,36.0,50.8,False,5.7,False
1,HAM,MERCEDES,2,90.294,1,SOFT,2,False,27.078,37.900,25.317,290.2,313.8,1,24.9,37.1,53.7,False,2.9,False
2,HAM,MERCEDES,3,90.233,1,SOFT,3,False,27.081,37.835,25.316,292.1,329.8,1,24.1,38.0,47.0,False,1.8,False
3,HAM,MERCEDES,4,90.382,1,SOFT,4,False,27.172,37.924,25.286,276.6,301.9,1,24.0,38.8,49.6,False,2.5,False
4,HAM,MERCEDES,5,90.774,1,SOFT,5,False,27.205,38.089,25.480,301.2,321.8,1,24.7,39.6,37.9,False,1.9,False


### 2.1 · Panel structure

The data is a *driver × lap* panel. Every feature below is computed **within driver,
ordered by lap**, so we establish and verify that ordering once, up front.

In [3]:
laps = laps.sort_values(["Driver", "LapNumber"]).reset_index(drop=True)

assert not laps.duplicated(["Driver", "LapNumber"]).any(), "Driver/LapNumber is not unique"
assert laps.notna().all().all(), "Task 4 should have delivered a complete frame"

TOTAL_LAPS = int(laps["LapNumber"].max())
DRIVERS = sorted(laps["Driver"].unique())

print(f"drivers   : {len(DRIVERS)}  {DRIVERS}")
print(f"race laps : {TOTAL_LAPS}")
print(f"compounds : {sorted(laps['Compound'].unique())}")
print(f"stints    : {sorted(laps['Stint'].unique())}")
print("\nlaps per driver (should be constant):")
print(laps.groupby("Driver")["LapNumber"].count().describe()[["min", "max"]].to_string())

drivers   : 10  ['ALO', 'GAS', 'HAM', 'LEC', 'NOR', 'PER', 'PIA', 'RUS', 'SAI', 'VER']
race laps : 55
compounds : ['HARD', 'MEDIUM', 'SOFT']
stints    : [np.int64(1), np.int64(2), np.int64(3)]

laps per driver (should be constant):
min    55.0
max    55.0


## 3 · Leakage and availability screen

Before engineering anything we classify **every raw column** into one of three roles.
This is the single most important step for model validity, so it is done explicitly and
auditable rather than by silent omission.

* **Predictor** — known *before* the lap is run, therefore legitimately usable.
* **Target** — the quantity to be predicted.
* **Excluded (leakage)** — only knowable *during or after* the lap in question.

The exclusions matter here:

| Column | Why excluded |
|---|---|
| `Sector1Time`, `Sector2Time`, `Sector3Time` | They sum **exactly** to `LapTime`. Task 4's correlation report already flagged this. Keeping them makes lap-time prediction a trivial identity. |
| `SpeedFL`, `SpeedST` | Same-lap telemetry — measured *while* the lap is being driven, so unavailable at prediction time. |
| `IsPersonalBest` | A deterministic function of `LapTime` versus the driver's earlier laps. |

`Driver`, `Team` and `Compound` are retained: they are known before the lap starts and are
encoded as features in §5.6.

In [4]:
sector_sum = laps[["Sector1Time", "Sector2Time", "Sector3Time"]].sum(axis=1)
residual = (laps["LapTime"] - sector_sum).abs()
print("Evidence for the sector-time exclusion")
print(f"  max |LapTime - sum(sectors)| = {residual.max():.4f} s")
print(f"  median                       = {residual.median():.2e} s")
print("  -> sectors reconstruct LapTime almost exactly; they are leakage, not features.\n")

LEAKAGE_COLS = ["Sector1Time", "Sector2Time", "Sector3Time",
                "SpeedFL", "SpeedST", "IsPersonalBest"]
TARGET_SOURCE_COLS = ["LapTime", "Stint"]

COLUMN_KIND = {c.name: c.kind.value for c in LAPS_SCHEMA.columns}

roles = []
for col in LAPS_SCHEMA.column_names():
    if col in LEAKAGE_COLS:
        role, note = "EXCLUDED (leakage)", "known only during/after the lap"
    elif col == "LapTime":
        role, note = "TARGET", "primary regression target"
    elif col == "Stint":
        role, note = "TARGET + predictor", "stint index; also encodes the pit event"
    else:
        role, note = "PREDICTOR", "known before the lap starts"
    roles.append({"column": col, "kind": COLUMN_KIND[col],
                  "role": role, "rationale": note})

column_roles = pd.DataFrame(roles)
column_roles

Evidence for the sector-time exclusion
  max |LapTime - sum(sectors)| = 2.8509 s
  median                       = 1.42e-14 s
  -> sectors reconstruct LapTime almost exactly; they are leakage, not features.



,column,kind,role,rationale
0,Driver,categorical,PREDICTOR,known before the lap starts
1,Team,categorical,PREDICTOR,known before the lap starts
2,LapNumber,numeric,PREDICTOR,known before the lap starts
3,LapTime,time,TARGET,primary regression target
4,Stint,numeric,TARGET + predictor,stint index; also encodes the pit event
5,Compound,categorical,PREDICTOR,known before the lap starts
6,TyreLife,numeric,PREDICTOR,known before the lap starts
7,FreshTyre,categorical,PREDICTOR,known before the lap starts
8,Sector1Time,time,EXCLUDED (leakage),known only during/after the lap
9,Sector2Time,time,EXCLUDED (leakage),known only during/after the lap


## 4 · Target definition

Three targets are derived, covering the modelling needs of Tasks 6–8. They are stored in the
export so downstream tasks select rather than recompute.

1. **`target_laptime`** — lap time in seconds. *Primary regression target*; drives the feature
   selection funnel below.
2. **`target_pit_next_lap`** — binary: does this driver pit at the end of this lap? Derived
   from the stint index incrementing on the following lap. Supports the strategy-decision
   classifier, and connects directly to the pit-stop rules of the Task 2 expert system.
3. **`target_laptime_fuel_corrected`** — lap time with the fuel-burn trend removed, isolating
   *tyre* degradation from the car simply getting lighter. This is the quantity Task 3's
   search cost model reasons about.

The fuel-burn coefficient is estimated from the data (regressing lap time on lap number while
controlling for tyre life and compound) rather than hard-coded, so it stays correct if the
notebook is pointed at a real FastF1 session.

In [5]:
grp = laps.groupby("Driver", sort=False)

# 1. lap time -------------------------------------------------------------
laps["target_laptime"] = laps["LapTime"].astype(float)

# 2. pit on this lap ------------------------------------------------------
next_stint = grp["Stint"].shift(-1)
laps["target_pit_next_lap"] = ((next_stint > laps["Stint"]) & next_stint.notna()).astype(int)

# 3. fuel-corrected lap time ---------------------------------------------
_design = pd.concat(
    [laps[["LapNumber", "TyreLife"]].astype(float),
     pd.get_dummies(laps["Compound"], prefix="cmp", dtype=float)],
    axis=1,
)
FUEL_COEF = float(LinearRegression().fit(_design, laps["target_laptime"]).coef_[0])
laps["target_laptime_fuel_corrected"] = (
    laps["target_laptime"] - FUEL_COEF * (laps["LapNumber"] - 1)
)

TARGETS = ["target_laptime", "target_pit_next_lap", "target_laptime_fuel_corrected"]

print(f"fuel-burn effect      : {FUEL_COEF:+.4f} s per lap "
      f"(negative = car speeds up as fuel burns off)")
print(f"pit events            : {int(laps['target_pit_next_lap'].sum())} "
      f"across {len(DRIVERS)} drivers")
print(f"lap time  mean/std    : {laps['target_laptime'].mean():.3f} / "
      f"{laps['target_laptime'].std():.3f} s")
laps[TARGETS].describe().round(3)

fuel-burn effect      : -0.0036 s per lap (negative = car speeds up as fuel burns off)
pit events            : 16 across 10 drivers
lap time  mean/std    : 91.255 / 0.596 s


,target_laptime,target_pit_next_lap,target_laptime_fuel_corrected
count,550.000,550.000,550.000
mean,91.255,0.029,91.350
std,0.596,0.168,0.616
min,89.519,0.000,89.519
25%,90.820,0.000,90.892
50%,91.229,0.000,91.327
75%,91.688,0.000,91.814
max,92.990,1.000,93.104


## 5 · Feature engineering

Six thematic blocks. Two rules govern all of them:

**(a) Causality.** Anything derived from race history uses `shift(1)` — only laps strictly
before the current one.

**(b) No exact linear identities.** If a feature is an exact linear combination of others
(e.g. `TyreLife` = the sum of its per-compound interactions, or `TrackTemp` = `AirTemp` +
their delta), the redundant member is *not constructed*. Otherwise the VIF step in §6.3
would be forced to choose arbitrarily among perfectly collinear columns and could delete the
physically meaningful variable rather than the derived one.

We also express pace in **gap-space** — each driver's lap time relative to the field median on
that lap — which separates "how fast is the track right now" from "how fast is this driver
relative to everyone else", instead of entangling them in one correlated block.

In [6]:
# Field-level pace reference, and each driver's gap to it (used by block C).
field_median = laps.groupby("LapNumber")["LapTime"].median()
laps["_field_median"] = laps["LapNumber"].map(field_median)
laps["_gap_to_field"] = laps["LapTime"] - laps["_field_median"]

F = pd.DataFrame(index=laps.index)
provenance: dict[str, str] = {}


def add(name: str, values, why: str) -> None:
    '''Register an engineered feature together with the reason it exists.'''
    F[name] = values
    provenance[name] = why


REFERENCE_COMPOUND = "HARD"   # baseline level for compound contrasts
print(f"reference compound for contrasts: {REFERENCE_COMPOUND}")

reference compound for contrasts: HARD


### 5.1 · Block A — Tyre and stint dynamics

The core physics of race strategy: a tyre loses grip as it ages, and it does so at a
compound-specific rate. Task 4's EDA measured that rate at roughly 0.088 / 0.047 / 0.033 s per
lap for SOFT / MEDIUM / HARD.

We deliberately **do not** feed those measured slopes in as a feature — they were estimated
from the target, so injecting them would be target leakage. Instead we supply
`tyre_life × compound` interaction terms and let the model *learn* the per-compound slope
itself. `HARD` is the reference level, so only SOFT and MEDIUM interactions are built; adding
a third would make `tyre_life` an exact sum of the three.

In [7]:
add("tyre_life", laps["TyreLife"].astype(float),
    "Laps completed on the current tyre set - the primary degradation driver.")
add("tyre_life_sq", laps["TyreLife"].astype(float) ** 2,
    "Quadratic term: degradation accelerates once the tyre passes its cliff.")
add("stint_number", laps["Stint"].astype(float),
    "Which stint of the race this is - proxies for race phase and strategy state.")
add("is_fresh_tyre", laps["FreshTyre"].astype(int),
    "Whether the set was new when fitted (scrubbed sets behave differently).")
add("tyrelife_x_soft", laps["TyreLife"] * (laps["Compound"] == "SOFT"),
    "Interaction: lets the model learn SOFT's steeper degradation slope vs HARD.")
add("tyrelife_x_medium", laps["TyreLife"] * (laps["Compound"] == "MEDIUM"),
    "Interaction: MEDIUM degradation slope relative to the HARD baseline.")

F[["tyre_life", "tyre_life_sq", "stint_number", "is_fresh_tyre",
   "tyrelife_x_soft", "tyrelife_x_medium"]].describe().round(2)

,tyre_life,tyre_life_sq,stint_number,is_fresh_tyre,tyrelife_x_soft,tyrelife_x_medium
count,550.00,550.00,550.00,550.00,550.00,550.00
mean,11.51,179.46,1.81,0.05,4.61,4.82
std,6.87,183.30,0.75,0.21,7.23,7.48
min,1.00,1.00,1.00,0.00,0.00,0.00
25%,6.00,36.00,1.00,0.00,0.00,0.00
50%,11.00,121.00,2.00,0.00,0.00,0.00
75%,16.00,256.00,2.00,0.00,8.00,9.00
max,28.00,784.00,3.00,1.00,27.00,28.00


### 5.2 · Block B — Fuel load and race progress

Cars start heavy and get lighter, so pace improves through the race independently of tyres.
`LapNumber`, `race_progress`, `fuel_load_frac` and `laps_remaining` are all exact affine
transforms of one another, so exactly **one** canonical axis is constructed.

In [8]:
add("race_progress", laps["LapNumber"] / TOTAL_LAPS,
    "Fraction of the race completed; proxies fuel burn-off and race phase. "
    "Canonical axis - lap number / fuel load / laps remaining are affine transforms of it.")

print(F[["race_progress"]].describe().round(3).to_string())

       race_progress
count        550.000
mean           0.509
std            0.289
min            0.018
25%            0.255
50%            0.509
75%            0.764
max            1.000


### 5.3 · Block C — Own recent pace (causal, in gap-space)

How the driver has actually been performing in the laps immediately before this one. All four
features are built from `gap_to_field`, **shifted by one lap**, so they encode only
information a race engineer would already have on the pit wall.

Working in gap-space rather than raw lap time keeps this block from being collinear with the
field-level pace in Block D.

Note we build `form_vs_baseline` (recent gap minus the driver's running average gap) and *not*
the raw lagged gap, because `gap_lag1 = form_vs_baseline + gap_expanding` would be an exact
identity.

In [9]:
gap_shift = laps.groupby("Driver", sort=False)["_gap_to_field"].shift(1)
by_driver = gap_shift.groupby(laps["Driver"])

_roll3_mean = by_driver.rolling(3, min_periods=3).mean().reset_index(level=0, drop=True)
_roll3_std = by_driver.rolling(3, min_periods=3).std().reset_index(level=0, drop=True)
_expanding = by_driver.expanding(min_periods=1).mean().reset_index(level=0, drop=True)

add("gap_roll3_mean", _roll3_mean,
    "Mean gap to the field over the previous 3 laps - short-run pace.")
add("gap_roll3_std", _roll3_std,
    "Volatility of that gap - unstable laps often precede a tyre cliff or traffic.")
add("gap_expanding", _expanding,
    "Running mean gap over all previous laps - the driver's baseline competitiveness.")
add("form_vs_baseline", gap_shift - _expanding,
    "Last lap's gap relative to the driver's own baseline: are they gaining or losing form?")

print("NaNs before the warm-up trim (expected - the first laps have no history):")
print(F[["gap_roll3_mean", "gap_roll3_std", "gap_expanding", "form_vs_baseline"]]
      .isna().sum().to_string())

NaNs before the warm-up trim (expected - the first laps have no history):
gap_roll3_mean      30
gap_roll3_std       30
gap_expanding       10
form_vs_baseline    10


### 5.4 · Block D — Field-level pace

The session-wide pace level and its trend. Together with Block C this decomposes each lap into
*"where is the whole field"* plus *"where is this driver relative to it"*.

In [10]:
add("field_median_lag1", laps["LapNumber"].sub(1).map(field_median),
    "Field median lap time on the previous lap - the ambient pace level "
    "(absorbs track evolution, safety cars, fuel effects common to everyone).")
add("field_pace_trend",
    laps["LapNumber"].sub(1).map(field_median) - laps["LapNumber"].sub(2).map(field_median),
    "Change in field median between the last two laps - is the track speeding up or slowing?")

print(F[["field_median_lag1", "field_pace_trend"]].describe().round(3).to_string())

       field_median_lag1  field_pace_trend
count            540.000           530.000
mean              91.221             0.029
std                0.372             0.191
min               90.254            -0.858
25%               90.983            -0.012
50%               91.245             0.064
75%               91.520             0.133
max               91.873             0.285


### 5.5 · Block E — Environment

Track temperature was the dominant environmental factor in Task 4's EDA. We supply `air_temp`
and the **track-minus-air delta** rather than air and track temperature separately, since the
three form an exact identity.

`tracktemp_dev_x_tyrelife` is the physically motivated interaction — hot tracks accelerate
degradation. Track temperature is mean-centred before multiplying so the interaction is not
simply a rescaled copy of `tyre_life`.

In [11]:
track_temp = laps["TrackTemp"].astype(float)
air_temp = laps["AirTemp"].astype(float)

add("air_temp", air_temp, "Ambient air temperature (deg C).")
add("track_air_delta", track_temp - air_temp,
    "Track minus air temperature - how much the surface has heated beyond ambient. "
    "Paired with air_temp instead of raw track temp to avoid an exact identity.")
add("humidity", laps["Humidity"].astype(float), "Relative humidity (%).")
add("wind_speed", laps["WindSpeed"].astype(float), "Wind speed (m/s); affects aero balance.")
add("tracktemp_dev_x_tyrelife", (track_temp - track_temp.mean()) * laps["TyreLife"],
    "Interaction: degradation accelerates on a hotter-than-average surface. "
    "Track temp is mean-centred so this is not a rescaled tyre_life.")

F[["air_temp", "track_air_delta", "humidity", "wind_speed",
   "tracktemp_dev_x_tyrelife"]].describe().round(2)

,air_temp,track_air_delta,humidity,wind_speed,tracktemp_dev_x_tyrelife
count,550.00,550.00,550.00,550.00,550.00
mean,23.99,13.91,53.43,3.20,0.14
std,1.41,2.89,10.10,1.73,33.63
min,20.00,5.80,35.10,0.00,-190.43
25%,23.00,11.80,45.40,1.80,-15.39
50%,24.00,14.00,53.40,3.30,0.70
75%,25.00,15.90,62.28,4.80,14.47
max,28.00,22.30,70.00,6.00,134.37


### 5.6 · Block F — Categorical encoding and constant-column candidates

Compound, team and driver are one-hot encoded with a **reference level dropped** (the first
category alphabetically), which avoids the dummy-variable trap and keeps the design matrix
full rank.

We also deliberately pass through `TrackStatus` and `Rainfall`. Task 4 established that both
are constant in this session, so they carry zero information — including them here lets the
near-zero-variance filter in §6.1 visibly do its job rather than being a no-op.

In [12]:
add("compound_soft", (laps["Compound"] == "SOFT").astype(int),
    f"Compound contrast vs the {REFERENCE_COMPOUND} reference level.")
add("compound_medium", (laps["Compound"] == "MEDIUM").astype(int),
    f"Compound contrast vs the {REFERENCE_COMPOUND} reference level.")

for team in sorted(laps["Team"].unique())[1:]:          # first team = reference
    key = f"team_{team.lower().replace(' ', '_')}"
    add(key, (laps["Team"] == team).astype(int), "Team identity (car performance).")

for drv in sorted(laps["Driver"].unique())[1:]:         # first driver = reference
    add(f"driver_{drv.lower()}", (laps["Driver"] == drv).astype(int),
        "Driver identity (skill / style).")

add("track_status", laps["TrackStatus"].astype(float),
    "Track status flag - constant in this session; retained so the NZV filter can remove it.")
add("is_rainfall", laps["Rainfall"].astype(int),
    "Rain flag - constant in this session; retained so the NZV filter can remove it.")

F = F.astype(float)
print(f"engineered features: {F.shape[1]}")
print(f"  reference team   : {sorted(laps['Team'].unique())[0]}")
print(f"  reference driver : {sorted(laps['Driver'].unique())[0]}")

engineered features: 36
  reference team   : ALPINE
  reference driver : ALO


### 5.7 · Warm-up trim

The rolling and expanding features need three prior laps before they are defined. Rather than
imputing values that never existed — which would fabricate history and bias the early race —
we drop the affected opening laps for each driver and record how many.

In [13]:
HISTORY_COLS = ["gap_roll3_mean", "gap_roll3_std", "form_vs_baseline",
                "field_median_lag1", "field_pace_trend"]

complete = F[HISTORY_COLS].notna().all(axis=1)
n_dropped = int((~complete).sum())

F = F[complete].reset_index(drop=True)
laps = laps[complete].reset_index(drop=True)

assert F.notna().all().all(), "unexpected NaNs after the warm-up trim"

WARMUP_LAPS = int(laps["LapNumber"].min())
print(f"dropped {n_dropped} warm-up rows (first {WARMUP_LAPS - 1} laps of each driver)")
print(f"modelling rows: {len(F)}  |  features: {F.shape[1]}")

dropped 30 warm-up rows (first 3 laps of each driver)
modelling rows: 520  |  features: 36


## 6 · Feature selection

A four-stage funnel, applied against the primary regression target `target_laptime`. Each
stage removes a *different* kind of unwanted feature, so the order matters:

| Stage | Removes | Rationale |
|---|---|---|
| 6.1 Near-zero variance | Constant / near-constant columns | Cannot inform any model |
| 6.2 Pairwise correlation | One of each redundant pair (\|r\| > 0.95) | Keeps the more informative twin |
| 6.3 Variance inflation | Multi-way collinearity (VIF > 10) | Pairwise checks miss 3-way redundancy |
| 6.4 Importance + stability | Weak / unstable predictors | Consensus across three different criteria |

Finally §6.5 chooses how many features to keep by measuring cross-validated error, and §6.6
verifies the reduced set against the full set.

### 6.0 · Validation strategy

Because this is a **time-ordered panel**, a random K-fold split would train on lap 50 and test
on lap 10 — leaking the future into the past and flattering every score.

Instead we use an **expanding-window, lap-forward split**: train on laps 1..k, test on the
next block of laps, then expand. Whole laps are kept together in a single fold so that the
field-median features cannot leak across the boundary. This mirrors how the model would
actually be used — predicting the remainder of a race from what has happened so far.

In [14]:
def lap_forward_splits(lap_numbers: pd.Series, n_splits: int = 4,
                       min_train_frac: float = 0.4):
    '''Expanding-window splits over lap number; whole laps stay in one fold.'''
    unique_laps = np.sort(lap_numbers.unique())
    bounds = np.linspace(int(len(unique_laps) * min_train_frac),
                         len(unique_laps), n_splits + 1).astype(int)
    for i in range(n_splits):
        train_laps = unique_laps[: bounds[i]]
        test_laps = unique_laps[bounds[i]: bounds[i + 1]]
        if len(test_laps) == 0:
            continue
        yield (np.where(lap_numbers.isin(train_laps))[0],
               np.where(lap_numbers.isin(test_laps))[0])


SPLITS = list(lap_forward_splits(laps["LapNumber"]))
y_reg = laps["target_laptime"].to_numpy()
y_clf = laps["target_pit_next_lap"].to_numpy()

for i, (tr, te) in enumerate(SPLITS, 1):
    print(f"fold {i}: train {len(tr):>3} rows (laps <= {laps['LapNumber'].iloc[tr].max():>2})"
          f"  ->  test {len(te):>3} rows "
          f"(laps {laps['LapNumber'].iloc[te].min()}-{laps['LapNumber'].iloc[te].max()})")

fold 1: train 200 rows (laps <= 23)  ->  test  80 rows (laps 24-31)
fold 2: train 280 rows (laps <= 31)  ->  test  80 rows (laps 32-39)
fold 3: train 360 rows (laps <= 39)  ->  test  80 rows (laps 40-47)
fold 4: train 440 rows (laps <= 47)  ->  test  80 rows (laps 48-55)


### 6.1 · Stage 1 — Near-zero variance

Drop columns that are constant, or whose most common value covers 99%+ of rows.

In [15]:
def near_zero_variance(frame: pd.DataFrame, freq_cut: float = 0.99) -> list[str]:
    dropped = []
    for col in frame.columns:
        top_share = frame[col].value_counts(normalize=True).iloc[0]
        if frame[col].nunique() <= 1 or top_share >= freq_cut:
            dropped.append(col)
    return dropped


nzv_dropped = near_zero_variance(F)
F_nzv = F.drop(columns=nzv_dropped)

print(f"dropped {len(nzv_dropped)}: {nzv_dropped}")
print(f"remaining: {F_nzv.shape[1]}")

dropped 2: ['track_status', 'is_rainfall']
remaining: 34


### 6.2 · Stage 2 — Pairwise correlation pruning

For any pair with |r| > 0.95 we keep whichever member has the higher mutual information
with the target, and drop the other.

In [16]:
mi_scores = pd.Series(
    mutual_info_regression(F_nzv, y_reg, random_state=RANDOM_STATE),
    index=F_nzv.columns,
)

corr = F_nzv.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

corr_dropped, corr_pairs = [], []
for col in upper.columns:
    for row in upper.index:
        r = upper.loc[row, col]
        if pd.notna(r) and r > 0.95:
            loser = col if mi_scores[col] < mi_scores[row] else row
            keeper = row if loser == col else col
            if loser not in corr_dropped:
                corr_dropped.append(loser)
                corr_pairs.append({"kept": keeper, "dropped": loser, "|r|": round(r, 4)})

F_corr = F_nzv.drop(columns=corr_dropped)
print(f"dropped {len(corr_dropped)} | remaining {F_corr.shape[1]}")
pd.DataFrame(corr_pairs) if corr_pairs else "no pairs above the 0.95 threshold"

dropped 1 | remaining 33


,kept,dropped,|r|
0,tyre_life,tyre_life_sq,0.9618


### 6.3 · Stage 3 — Variance inflation factors

Pairwise correlation cannot detect a feature that is redundant given a *combination* of
others. VIF measures exactly that: regress each feature on all the rest and compute
$\text{VIF} = 1/(1-R^2)$. We drop the worst offender above 10 and repeat.

VIF is applied **only to continuous features**. One-hot dummies are structurally correlated
with each other and routinely show inflated VIFs even when perfectly well behaved, so
including them would prune valid categorical information.

In [17]:
binary_cols = [c for c in F_corr.columns if F_corr[c].isin([0, 1]).all()]
continuous_cols = [c for c in F_corr.columns if c not in binary_cols]


def prune_by_vif(frame: pd.DataFrame, cols: list[str], threshold: float = 10.0):
    cols = list(cols)
    Z = pd.DataFrame(StandardScaler().fit_transform(frame[cols]), columns=cols)
    removed = []
    while len(cols) > 2:
        vifs = {}
        for c in cols:
            others = [x for x in cols if x != c]
            r2 = LinearRegression().fit(Z[others], Z[c]).score(Z[others], Z[c])
            vifs[c] = 1.0 / max(1e-12, 1.0 - r2)
        worst = max(vifs, key=vifs.get)
        if vifs[worst] <= threshold:
            break
        cols.remove(worst)
        removed.append({"dropped": worst, "VIF": round(min(vifs[worst], 1e6), 1)})
    return cols, removed


kept_continuous, vif_removed = prune_by_vif(F_corr, continuous_cols)
F_sel = F_corr[kept_continuous + binary_cols]

print(f"continuous in: {len(continuous_cols)} -> kept {len(kept_continuous)}")
print(f"binary/dummy (exempt): {len(binary_cols)}")
print(f"remaining: {F_sel.shape[1]}")
pd.DataFrame(vif_removed) if vif_removed else "no feature exceeded VIF 10"

continuous in: 16 -> kept 14
binary/dummy (exempt): 17
remaining: 31


,dropped,VIF
0,stint_number,38.0
1,tyrelife_x_soft,12.7


### 6.4 · Stage 4 — Importance ranking and stability

Any single importance measure has a bias: mutual information sees non-linear but marginal
relationships, tree importance favours high-cardinality continuous features, and L1 regression
sees only linear effects. We therefore rank by **all three** and average the ranks, so a
feature must convince more than one criterion.

We then re-run the ranking **inside every training fold** and record how often each feature
lands in the top 12. A feature that is important in one fold and irrelevant in the next is
fitting noise, not signal.

In [18]:
def rank_features(X: pd.DataFrame, y: np.ndarray, task: str = "reg") -> pd.DataFrame:
    '''Rank features by mutual information, tree importance and L1 coefficient.'''
    Xs = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)
    if task == "reg":
        mi = mutual_info_regression(X, y, random_state=RANDOM_STATE)
        forest = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE,
                                       n_jobs=-1).fit(X, y)
        linear = np.abs(LassoCV(cv=5, random_state=RANDOM_STATE,
                                max_iter=5000).fit(Xs, y).coef_)
    else:
        mi = mutual_info_classif(X, y, random_state=RANDOM_STATE)
        forest = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE,
                                        class_weight="balanced", n_jobs=-1).fit(X, y)
        linear = np.abs(LogisticRegression(penalty="l1", solver="liblinear", C=0.5,
                                           class_weight="balanced",
                                           max_iter=5000).fit(Xs, y).coef_[0])
    out = pd.DataFrame({"mutual_info": mi, "tree_importance": forest.feature_importances_,
                        "l1_coef": linear}, index=X.columns)
    out["avg_rank"] = out.rank(ascending=False).mean(axis=1)
    return out.sort_values("avg_rank")


ranking = rank_features(F_sel, y_reg)

TOP_K_STABILITY = 12
hits = pd.Series(0, index=F_sel.columns, dtype=int)
for train_idx, _ in SPLITS:
    fold_rank = rank_features(F_sel.iloc[train_idx], y_reg[train_idx])
    hits[fold_rank.head(TOP_K_STABILITY).index] += 1
ranking["stability"] = hits / len(SPLITS)

ranking.head(15).round(4)

,mutual_info,tree_importance,l1_coef,avg_rank,stability
tyre_life,0.2866,0.3491,0.2477,2.0000,1.00
gap_roll3_mean,0.3542,0.1073,0.1141,2.3333,1.00
gap_expanding,0.2976,0.3146,0.1566,2.3333,1.00
field_median_lag1,0.1591,0.0444,0.0727,4.3333,1.00
driver_sai,0.0776,0.0207,0.0065,8.3333,1.00
race_progress,0.3104,0.0430,0.0000,8.6667,1.00
tracktemp_dev_x_tyrelife,0.1230,0.0104,0.0000,11.6667,1.00
form_vs_baseline,0.0976,0.0185,0.0000,12.0000,1.00
tyrelife_x_medium,0.1511,0.0090,0.0000,12.3333,0.75
gap_roll3_std,0.0402,0.0134,0.0000,15.0000,0.50


### 6.5 · Choosing how many features to keep

Rather than picking a round number, we sweep K and measure cross-validated error using the
lap-forward splits, then apply a **parsimony rule**: keep the smallest K whose mean absolute
error is within 2% of the best observed. This trades a negligible amount of accuracy for a
markedly simpler model — which matters for Task 8, where every retained feature costs qubits.

In [19]:
def cv_regression(cols: list[str]) -> tuple[float, float]:
    maes, r2s = [], []
    for train_idx, test_idx in SPLITS:
        model = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
        model.fit(F_sel.iloc[train_idx][cols], y_reg[train_idx])
        pred = model.predict(F_sel.iloc[test_idx][cols])
        maes.append(mean_absolute_error(y_reg[test_idx], pred))
        r2s.append(r2_score(y_reg[test_idx], pred))
    return float(np.mean(maes)), float(np.mean(r2s))


ranked_order = ranking.index.tolist()
sweep_ks = [k for k in (4, 6, 8, 10, 12, 15, 20, len(ranked_order)) if k <= len(ranked_order)]
sweep = {k: cv_regression(ranked_order[:k]) for k in sorted(set(sweep_ks))}

sweep_df = pd.DataFrame(
    [{"K": k, "cv_MAE_s": round(m, 4), "cv_R2": round(r, 4)} for k, (m, r) in sweep.items()]
).set_index("K")

best_mae = sweep_df["cv_MAE_s"].min()
K_STAR = int(sweep_df.index[sweep_df["cv_MAE_s"] <= best_mae * 1.02].min())
selected_regression = ranked_order[:K_STAR]

print(sweep_df.to_string())
print(f"\nbest MAE {best_mae:.4f}s -> within 2% tolerance {best_mae * 1.02:.4f}s")
print(f"selected K* = {K_STAR}")
for i, f in enumerate(selected_regression, 1):
    print(f"  {i:>2}. {f}")

    cv_MAE_s   cv_R2
K                   
4     0.2477  0.4951
6     0.2413  0.5361
8     0.2447  0.5150
10    0.2476  0.5176
12    0.2495  0.5116
15    0.2498  0.5096
20    0.2427  0.5203
31    0.2447  0.5188

best MAE 0.2413s -> within 2% tolerance 0.2461s
selected K* = 6
   1. tyre_life
   2. gap_roll3_mean
   3. gap_expanding
   4. field_median_lag1
   5. driver_sai
   6. race_progress


### 6.6 · Does the reduced set hold up?

Compare the selected features against the full post-VIF set on the same folds. The reduced
set should match or beat the full set — if it does, the discarded features were noise.

In [20]:
mae_sel, r2_sel = cv_regression(selected_regression)
mae_all, r2_all = cv_regression(list(F_sel.columns))

comparison = pd.DataFrame({
    "n_features": [len(selected_regression), F_sel.shape[1]],
    "cv_MAE_s": [round(mae_sel, 4), round(mae_all, 4)],
    "cv_R2": [round(r2_sel, 4), round(r2_all, 4)],
}, index=["selected", "all (post-VIF)"])

print(comparison.to_string())
verdict = ("selected set matches or beats the full set"
           if mae_sel <= mae_all else "full set is stronger - review K*")
print(f"\n{len(selected_regression)}/{F_sel.shape[1]} features retained "
      f"({1 - len(selected_regression) / F_sel.shape[1]:.0%} reduction); {verdict}.")

                n_features  cv_MAE_s   cv_R2
selected                 6    0.2413  0.5361
all (post-VIF)          31    0.2450  0.5180

6/31 features retained (81% reduction); selected set matches or beats the full set.


### 6.7 · Feature selection for the pit-decision target

The same machinery, applied to `target_pit_next_lap`, so Task 6's classifier gets its own
selected set rather than inheriting one tuned for regression.

**Read the score below with caution.** Pit events are rare (a handful out of several hundred
laps), and in the synthetic session that backs this pipeline stint lengths follow a
near-deterministic schedule — so tyre age alone almost perfectly determines the pit lap. The
AUC reported here therefore reflects the *data generator*, not the genuine difficulty of race
strategy. On a real FastF1 session, with safety cars and reactive undercuts, expect a
substantially lower and more interesting score. The feature *list* remains the useful output.

In [21]:
ranking_clf = rank_features(F_sel, y_clf, task="clf")


def cv_classification(cols: list[str]) -> float:
    aucs = []
    for train_idx, test_idx in SPLITS:
        if len(np.unique(y_clf[test_idx])) < 2:
            continue
        model = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE,
                                       class_weight="balanced", n_jobs=-1)
        model.fit(F_sel.iloc[train_idx][cols], y_clf[train_idx])
        proba = model.predict_proba(F_sel.iloc[test_idx][cols])[:, 1]
        aucs.append(roc_auc_score(y_clf[test_idx], proba))
    return float(np.mean(aucs)) if aucs else float("nan")


clf_order = ranking_clf.index.tolist()
clf_sweep = {k: cv_classification(clf_order[:k]) for k in (4, 6, 8, 10)}
K_CLF = 8
selected_classification = clf_order[:K_CLF]

print(f"class balance: {int(y_clf.sum())} pit events / {len(y_clf)} laps "
      f"({y_clf.mean():.1%})\n")
print("cv AUC by K: " + ", ".join(f"K={k}: {v:.3f}" for k, v in clf_sweep.items()))
print(f"\nselected ({K_CLF}): {selected_classification}")
ranking_clf.head(10).round(4)

class balance: 16 pit events / 520 laps (3.1%)

cv AUC by K: K=4: 1.000, K=6: 1.000, K=8: 0.993, K=10: 0.982

selected (8): ['tyre_life', 'race_progress', 'form_vs_baseline', 'field_median_lag1', 'team_aston_martin', 'field_pace_trend', 'tyrelife_x_medium', 'tracktemp_dev_x_tyrelife']


,mutual_info,tree_importance,l1_coef,avg_rank
tyre_life,0.0912,0.2722,5.4602,1.6667
race_progress,0.0930,0.0752,1.5967,4.0000
form_vs_baseline,0.0175,0.1246,0.9107,5.0000
field_median_lag1,0.0922,0.0984,0.5386,5.0000
team_aston_martin,0.0113,0.0034,2.3706,9.0000
field_pace_trend,0.0856,0.1174,0.0000,10.3333
tyrelife_x_medium,0.0427,0.0387,0.0000,12.3333
tracktemp_dev_x_tyrelife,0.0000,0.0456,0.1237,13.3333
humidity,0.0000,0.0278,0.1292,14.3333
driver_ham,0.0000,0.0028,2.4458,14.6667


## 7 · Export

Two files, and nothing else.

**`f1_features_selected.csv`** — identifier columns, the union of both selected feature sets,
and all three targets. The union is exported (rather than two separate files) so Tasks 6–8
read one dataset and pick columns via the metadata.

**`feature_metadata.json`** — the contract for downstream tasks: which columns are features
for which target, what each feature means and why it exists, which columns need scaling, and
the validation strategy that must be reused to keep results comparable.

A note on what is deliberately *not* exported: the features are **unscaled**. Standardising
here would compute means and standard deviations over the whole dataset, leaking test-fold
statistics into training. Tasks 6–8 should fit the scaler inside their own CV pipeline —
`numeric_features_requiring_scaling` in the metadata lists exactly which columns need it.

In [22]:
ID_COLS = ["Driver", "LapNumber", "Stint", "Compound", "Team"]
selected_union = sorted(set(selected_regression) | set(selected_classification))

export = pd.concat(
    [laps[ID_COLS].reset_index(drop=True),
     F[selected_union].reset_index(drop=True),
     laps[TARGETS].reset_index(drop=True)],
    axis=1,
)

assert export.notna().all().all(), "export contains NaNs"
assert not export.columns.duplicated().any(), "duplicate columns in export"
for leak in LEAKAGE_COLS:
    assert leak not in export.columns, f"leakage column {leak} reached the export"

csv_path = OUT_DIR / "f1_features_selected.csv"
export.to_csv(csv_path, index=False)

binary_selected = [c for c in selected_union if F[c].isin([0, 1]).all()]
metadata = {
    "task": "Phase 2 / Task 5 - Feature Engineering & Feature Selection",
    "source_dataset": str(CLEAN_CSV.relative_to(ROOT)),
    "rows": int(len(export)),
    "random_state": RANDOM_STATE,
    "identifier_columns": ID_COLS,
    "targets": {
        "target_laptime": "Lap time in seconds - primary regression target (Task 6).",
        "target_pit_next_lap": "1 if the driver pits at the end of this lap - classification.",
        "target_laptime_fuel_corrected":
            f"Lap time with the fuel-burn trend ({FUEL_COEF:+.4f} s/lap) removed, "
            "isolating tyre degradation.",
    },
    "selected_features": {
        "target_laptime": selected_regression,
        "target_pit_next_lap": selected_classification,
        "union_exported": selected_union,
    },
    "feature_provenance": {f: provenance[f] for f in selected_union},
    "numeric_features_requiring_scaling":
        [c for c in selected_union if c not in binary_selected],
    "binary_features_no_scaling_needed": binary_selected,
    "preprocessing_contract": {
        "scaling": "NOT applied here. Fit the scaler inside each CV fold to avoid leakage.",
        "validation": "Expanding-window lap-forward split; keep whole laps in one fold. "
                      "Do not use random K-fold - this is a time-ordered panel.",
        "warmup_rows_dropped": n_dropped,
        "first_usable_lap": WARMUP_LAPS,
    },
    "selection_funnel": {
        "1_near_zero_variance_dropped": nzv_dropped,
        "2_correlation_dropped": corr_dropped,
        "3_vif_dropped": [d["dropped"] for d in vif_removed],
        "4_importance_and_stability": f"ranked by mutual information + tree importance + "
                                      f"L1 coefficient; K*={K_STAR} chosen within 2% of best CV MAE",
    },
    "excluded_as_leakage": {
        "columns": LEAKAGE_COLS,
        "reason": "Sector times sum exactly to LapTime; speed traps and IsPersonalBest are "
                  "only knowable during or after the lap being predicted.",
    },
    "validation_scores": {
        "regression_cv_MAE_s": round(mae_sel, 4),
        "regression_cv_R2": round(r2_sel, 4),
        "classification_cv_AUC": round(clf_sweep[K_CLF], 4),
        "caveat": "Classification AUC is inflated by the near-deterministic stint schedule "
                  "of the synthetic session; expect lower values on real FastF1 data.",
    },
}

json_path = OUT_DIR / "feature_metadata.json"
json_path.write_text(json.dumps(metadata, indent=2))

print(f"wrote {csv_path.relative_to(ROOT)}  ({export.shape[0]} x {export.shape[1]})")
print(f"wrote {json_path.relative_to(ROOT)}")
print(f"\ncolumns: {list(export.columns)}")
export.head()

wrote phase2_task5_feature_engineering/outputs/f1_features_selected.csv  (520 x 19)
wrote phase2_task5_feature_engineering/outputs/feature_metadata.json

columns: ['Driver', 'LapNumber', 'Stint', 'Compound', 'Team', 'driver_sai', 'field_median_lag1', 'field_pace_trend', 'form_vs_baseline', 'gap_expanding', 'gap_roll3_mean', 'race_progress', 'team_aston_martin', 'tracktemp_dev_x_tyrelife', 'tyre_life', 'tyrelife_x_medium', 'target_laptime', 'target_pit_next_lap', 'target_laptime_fuel_corrected']


,Driver,LapNumber,Stint,Compound,Team,driver_sai,field_median_lag1,field_pace_trend,form_vs_baseline,gap_expanding,gap_roll3_mean,race_progress,team_aston_martin,tracktemp_dev_x_tyrelife,tyre_life,tyrelife_x_medium,target_laptime,target_pit_next_lap,target_laptime_fuel_corrected
0,ALO,4,1,SOFT,ASTON MARTIN,0.0,90.5515,0.1645,0.144000,0.005500,0.005500,0.072727,1.0,-17.204364,4.0,0.0,90.943,0,90.953660
1,ALO,5,1,SOFT,ASTON MARTIN,0.0,90.5150,-0.0365,0.316875,0.111125,0.122500,0.090909,1.0,-22.505455,5.0,0.0,90.879,0,90.893213
2,ALO,6,1,SOFT,ASTON MARTIN,0.0,90.7130,0.1980,0.043900,0.122100,0.247833,0.109091,1.0,-28.206545,6.0,0.0,90.846,0,90.863766
3,ALO,7,1,SOFT,ASTON MARTIN,0.0,90.7765,0.0635,-0.043833,0.113333,0.221167,0.127273,1.0,15.392364,7.0,0.0,91.051,0,91.072320
4,ALO,8,1,SOFT,ASTON MARTIN,0.0,90.7605,-0.0160,0.151857,0.138643,0.175333,0.145455,1.0,-11.208727,8.0,0.0,90.947,0,90.971873


## 8 · Final verification

Re-read both files from disk exactly as Task 6 would, and confirm they are usable.

In [23]:
check_df = pd.read_csv(csv_path)
check_meta = json.loads(json_path.read_text())

feat_reg = check_meta["selected_features"]["target_laptime"]
X_check = check_df[feat_reg]
y_check = check_df["target_laptime"]

assert list(check_df.columns) == list(export.columns)
assert X_check.notna().all().all()
assert len(check_df) == metadata["rows"]

print("round-trip verification")
print(f"  rows / columns        : {check_df.shape[0]} x {check_df.shape[1]}")
print(f"  regression features   : {len(feat_reg)}")
print(f"  classification feats  : {len(check_meta['selected_features']['target_pit_next_lap'])}")
print(f"  needs scaling         : {len(check_meta['numeric_features_requiring_scaling'])}")
print(f"  targets               : {list(check_meta['targets'])}")
print(f"  files written         : 2")
print("\nTask 5 complete - ready for Task 6 (Classical Machine Learning).")

round-trip verification
  rows / columns        : 520 x 19
  regression features   : 6
  classification feats  : 8
  needs scaling         : 9
  targets               : ['target_laptime', 'target_pit_next_lap', 'target_laptime_fuel_corrected']
  files written         : 2

Task 5 complete - ready for Task 6 (Classical Machine Learning).


## 9 · Summary and hand-off to Task 6

**What this notebook did**

1. Read Task 4's cleaned per-lap dataset in place — no dataset was copied or duplicated.
2. Excluded six columns as leakage, with the sector-time identity demonstrated numerically
   rather than asserted.
3. Engineered features across six blocks — tyre/stint dynamics, fuel and race progress, causal
   pace history in gap-space, field-level pace, environment, and reference-encoded
   categoricals — on a basis chosen to contain no exact linear identities.
4. Derived three targets covering regression, classification, and fuel-corrected pace.
5. Ran a four-stage selection funnel (near-zero variance → correlation → VIF → importance with
   fold stability) and chose the feature count by cross-validated error under a parsimony rule.
6. Exported one dataset and one metadata file.

**How Task 6 should consume this**

```python
import json, pandas as pd

df = pd.read_csv("../phase2_task5_feature_engineering/outputs/f1_features_selected.csv")
meta = json.loads(open("../phase2_task5_feature_engineering/outputs/feature_metadata.json").read())

X = df[meta["selected_features"]["target_laptime"]]
y = df["target_laptime"]
# Fit the scaler inside each CV fold - see meta["preprocessing_contract"].
```

**Two things to carry forward honestly**

* The pipeline currently runs on Task 4's synthetic session, which reproduces the real Kaggle
  and FastF1 schemas but has a near-deterministic pit schedule. That inflates the pit-decision
  AUC specifically. Pointing Task 4 at real CSVs flows through to this notebook unchanged.
* Features are exported unscaled by design. Tasks 7 and 8 in particular should scale within
  their pipelines — and for the quantum models, `K*` was kept deliberately small because each
  retained feature translates into circuit width.